In [1]:
import langchain
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate


os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model(
    "llama-3.1-8b-instant", 
    model_provider="groq"
)


c:\Users\91766\Downloads\invitationn\projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Document loaders
1) textLoader
2) PyPDFLoader


In [2]:
#1) textLoader
from langchain_community.document_loaders import TextLoader

data = TextLoader("document loader/notes.txt")
docs = data.load()

print(docs)


C:\Users\91766\AppData\Local\Temp\ipykernel_4112\631796051.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'document loader/notes.txt'}, page_content='Blue bicycles race down the steep hill as the golden sun starts to set. A tiny brown cat watches them from a warm stone wall, purring softly.Daily AdventuresThe old clock ticks loud in the quiet room.Fresh green leaves dance in the cool morning wind.A small yellow bird sings a happy song.If you want, tell me:What topic or theme do you want?How many sentences do you need?I can write a new paragraph just for you.')]


In [3]:
#2) PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader
data = PyPDFLoader("document loader\\Your paragraph text.pdf")
docs = data.load()

print(docs)

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2025-04-29T14:16:54+00:00', 'title': 'Your paragraph text', 'moddate': '2025-04-29T14:16:54+00:00', 'keywords': 'DAGmBUjJukY,BAEeCwISFNE,0', 'author': 'Retort Conundrum', 'source': 'document loader\\Your paragraph text.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='REBATE POLICY FOR MESS \nMinimum Rebate Duration:\nA rebate will be granted only for absences of 5 days or more.\nRequests for rebates for periods shorter than 5 days will not be\nconsidered under any circumstances.\nMandatory Rebate Entry/Exit Registration:\nAll residents must record both their entry and exit dates in the\nRebate Register to be eligible for a mess rebate. \nNo other forms of documentation (such as travel tickets or gate entry\nlogs) will be accepted as valid proof.\nAccuracy and Honesty:\nSubmitting false or misleading information in the Rebate Register is a\nserious offense and may result in strict disciplin

In [4]:
#3) Implementation
template = ChatPromptTemplate.from_messages([
    ("system", "You are an AI that summarizes text. "),
    ("human", "{data}")
])

prompt = template.format_messages(data = docs[0].page_content)


result = model.invoke(prompt)

print(result.content)

The Mess Rebate Policy outlines the following key points:

1. **Minimum Rebate Duration**: A rebate is granted only for absences of 5 days or more.
2. **Mandatory Entry/Exit Registration**: Residents must record their entry and exit dates in the Rebate Register to be eligible for a rebate.
3. **Accuracy and Honesty**: Submitting false information is a serious offense and may result in disciplinary action.
4. **Missed Rebate Entry**: If unable to make an entry personally, a fellow resident or friend can record the rebate on behalf of the individual, but the individual is still responsible for ensuring accuracy.
5. **Extension of Rebate Period**: Residents can extend their rebate period due to unforeseen circumstances, but must provide valid supporting documents.
6. **No Retroactive Rebates**: Rebates will not be granted for periods where no entry/exit has been made in the register, regardless of the reason.

The policy aims to maintain transparency, fairness, and cost efficiency for all

Most of the embedding models and language model have a context window and they cannot take infinite tokens at the same time so for that we have use to use text Splitting or Chunking 

In [5]:
sample = "Farmers were working hard in the fields, preparing the soil and planting seeds for the next season. The sun was bright, and the air smelled of earth and fresh grass. The Indian Premier League (IPL) is the biggest cricket league in the world. People all over the world watch the matches and cheer for their favourite teams.Terrorism is a big danger to peace and safety. It causes harm to people and creates fear in cities and villages. When such attacks happen, they leave behind pain and sadness. To fight terrorism, we need strong laws, alert security forces, and support from people who care about peace and safety."

In [6]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings # Replaced OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

# Initialize a free, local Hugging Face embedding model instead of OpenAI
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")



C:\Users\91766\AppData\Local\Temp\ipykernel_4112\2612215491.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 901.18it/s]


In [7]:
text_splitter = SemanticChunker(
    embedding_model, 
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=80 # Splits at the top 20% most drastically different sentences
)
sample = """
Farmers were working hard in the fields, preparing the soil and planting seeds for the next season. The sun was bright, and the air smelled of earth and fresh grass. The Indian Premier League (IPL) is the biggest cricket league in the world. People all over the world watch the matches and cheer for their favourite teams.

Terrorism is a big danger to peace and safety. It causes harm to people and creates fear in cities and villages. When such attacks happen, they leave behind pain and sadness. To fight terrorism, we need strong laws, alert security forces, and support from people who care about peace and safety.
"""

docs = text_splitter.create_documents([sample])
print(len(docs))
print(docs)

3
[Document(metadata={}, page_content='\nFarmers were working hard in the fields, preparing the soil and planting seeds for the next season.'), Document(metadata={}, page_content='The sun was bright, and the air smelled of earth and fresh grass. The Indian Premier League (IPL) is the biggest cricket league in the world. People all over the world watch the matches and cheer for their favourite teams. Terrorism is a big danger to peace and safety. It causes harm to people and creates fear in cities and villages. When such attacks happen, they leave behind pain and sadness. To fight terrorism, we need strong laws, alert security forces, and support from people who care about peace and safety.'), Document(metadata={}, page_content='')]


vector Store implementation 


vectore are not responsible for answering your query they retrieve your information only

In [8]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

docs = [
    Document(page_content="Python is widely used in Artificial Intelligence.", metadata={"source": "AI_book"}),
    Document(page_content="Pandas is used for data analysis in Python.", metadata={"source": "DataScience_book"}),
    Document(page_content="Neural networks are used in deep learning.", metadata={"source": "DL_book"}),
]

vectorstore = Chroma.from_documents( 
    documents = docs,
    embedding = embedding_model,
    persist_directory= "chroma_db"
)




Similarity Search 

K value means total how many answer you want !

In [9]:
result = vectorstore.similarity_search("What is used for deep learning ? ", k = 2) 
for r in result:
    print(r)


page_content='Neural networks are used in deep learning.' metadata={'source': 'DL_book'}
page_content='Python is widely used in Artificial Intelligence.' metadata={'source': 'AI_book'}


In [10]:
retriver = vectorstore.as_retriever()

docs = retriver.invoke("Explain deep learning")

for d in docs:
    print(d)

page_content='Neural networks are used in deep learning.' metadata={'source': 'DL_book'}
page_content='Python is widely used in Artificial Intelligence.' metadata={'source': 'AI_book'}
page_content='Pandas is used for data analysis in Python.' metadata={'source': 'DataScience_book'}


In [11]:
#load pdf 
#split into chunks 
#create the embeddings 
#store into chroma 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma 


data = PyPDFLoader("document loader\\100 Page ML.pdf")
docs = data.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = splitter.split_documents(docs)


vectorstore = Chroma.from_documents(
    documents= chunks,
    embedding=embedding_model,
    persist_directory="database"
)

In [12]:
result = vectorstore.similarity_search("what is KNN algorithm and how it is used ?")

for doc in result:
    print(doc.page_content)
    print("---")

algorithm frequently used in practice isk-Nearest Neighbors(kNN). In classiﬁcation, to
predict a label for an input example the kNN algorithm looks at the close neighborhood of
the input example in the space of feature vectors and outputs the label that it saw the most
often in this close neighborhood.
3There’s still one label per example though.
Andriy Burkov The Hundred-Page Machine Learning Book - Draft 13
---
d(xi, xk)
def
=
∏(
x(1)
i −x(1)
k
)2
+
(
x(2)
i −x(2)
k
)2
+···+
(
x(N )
i −x(N )
k
)2
=
√
D∑
j=1
(
x(j)
i −x(j)
k
)2
.
It can be shown that the feature space of the RBF (for “radial basis function”) kernel has
an inﬁnite number of dimensions. By varying the hyperparameterσ, the data analyst can
choose between getting a smooth or curvy decision boundary in the original space.
3.5 k-Nearest Neighbors
k-Nearest Neighbors(kNN) is a non-parametric learning algorithm. Contrary to other
learning algorithms that allow discarding the training data after the model is built, kNN
keep

Retrievers 

1 by datasource
2 by retrieval stragey

In [16]:
#1 datasource

from langchain_community.retrievers import ArxivRetriever

retriver = ArxivRetriever(
    load_max_docs= 2,
    load_all_available_meta= True
)

docs = retriver.invoke("large language model")

# print results
for i, doc in enumerate(docs):
    print(f"\nResult {i+1}")
    print("Title:", doc.metadata.get("Title"))
    print("Authors:", doc.metadata.get("Authors"))
    print("Summary:", doc.page_content)  # print first 500 characters


AttributeError: 'Search' object has no attribute 'results'